# Track C — VEGAS × entropy ablation (CS 639)

**This is the turn-key Colab notebook for Track C.** Run all cells top to bottom.

**Runtime profile**

| Accelerator | Config used by this notebook | Peak VRAM | Wall-time, 500 images × 2 passes |
|---|---|---|---|
| **H100 / A100** | fp16 LLaVA-1.5-7B, `attn_implementation="eager"` | ~18 GB | ~40 min |
| **T4 (free Colab)** | 4-bit NF4 LLM + fp16 ViT (bitsandbytes) | ~8 GB | ~2–3 h |
| TPU v5e-1 | not supported (torch_xla + forward hooks don't mix) | — | — |

Pick `USE_4BIT = True` in §3 if you're on a T4. The 4-bit path also re-runs the
**vanilla baseline** in-notebook, so the vanilla-vs-VEGAS comparison stays fair
(same weights, same quantization).

**What it does**

1. Install pinned deps (`transformers==4.37.2` — last LLaVA-1.5-compatible release).
2. Mount Drive, locate Track A outputs (`sampled_image_ids.json`, `entropy_ranks.csv`,
   `chair_labels.csv`) and the `trackc/` package.
3. Download the 500 sampled COCO val2017 images + `instances_val2017.json` (~155 MB).
4. Load LLaVA-1.5-7B (fp16 or 4-bit depending on the flag).
5. Run **VEGAS** — ViT final-layer [CLS] attention injected at LLM layers 14 & 15
   (paper: arXiv:2512.12089 §5.1) — on all 500 images → `captions_vegas.csv`.
6. On 4-bit runs, also re-run vanilla → `captions_vanilla_4bit.csv`.
7. Run the entropy-stratified CHAIR ablation in-notebook. Writes `ablation_summary.csv`,
   `vegas_minus_vanilla_delta.csv`, `bootstrap_delta.csv`.

Hand back `captions_vegas.csv` (and, on T4, `captions_vanilla_4bit.csv`) to the local
analysis step which produces `TrackC_Report.docx`.

## 1. Colab setup — installs, Drive mount, paths

In [ ]:
# Modern transformers + matching tokenizers. We tried pinning transformers==4.37.2
# but the LLaVA tokenizer.json on the Hub now requires tokenizers>=0.20, which
# that old transformers can't use. transformers>=4.47 handles LLaVA-1.5 cleanly
# and the VEGAS forward-pre-hook in trackc.vegas is compatible with it.
#
# IMPORTANT: after this cell finishes, do Runtime → Restart session, then
# re-run from cell 3 onward (skip this one).
!pip install -q -U "transformers>=4.47,<5" "tokenizers>=0.20" \
    "accelerate>=0.30" "bitsandbytes>=0.43" \
    "sentencepiece" "protobuf" "pycocotools" "python-docx" "tqdm" "Pillow"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# EDIT THIS if your Track A folder sits somewhere else in Drive.
# It must contain: sampled_image_ids.json, entropy_ranks.csv, chair_labels.csv,
# and the trackc/ package (from this repo).
import os, sys
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/TrackA_Complete_Outputs")     # <-- ADJUST IF NEEDED
assert ROOT.exists(), f"Track A folder not found at {ROOT}"
sys.path.insert(0, str(ROOT))

WORK = Path("/content/trackc_run")
WORK.mkdir(exist_ok=True)
(WORK / "results").mkdir(exist_ok=True)

print("Track A folder:", ROOT)
print("Contents:", sorted(p.name for p in ROOT.iterdir())[:20])

Mounted at /content/drive
Track A folder: /content/drive/MyDrive/TrackA_Complete_Outputs
Contents: ['.DS_Store', '.venv', 'TRACKA_1ST_INFERENCE_CHAIR.ipynb', 'TRACKC_VEGAS_Ablate.ipynb', 'TrackA_2ND_Attention_Rollout.ipynb', 'ViT_Hallucination_Research_Workflow.docx', 'attention_viz', 'chair_distribution.png', 'chair_labels.csv', 'coco', 'entropy_ranks.csv', 'figure_entropy_distribution.png', 'figure_per_layer_entropy.png', 'figure_quartile_analysis.png', 'figure_spearman_scatter.png', 'requirements-trackc.txt', 'rollout_data_complete.npz', 'run_trackc_report.py', 'sampled_image_ids.json', 'trackc']


## 2. Download the 500 sampled COCO val2017 images + annotations

Uses `http://images.cocodataset.org/val2017/<12-digit-id>.jpg` (public, unauthenticated).
Also downloads `instances_val2017.json` for CHAIR ground-truth recomputation.
Total download is ≈150 MB for images + ~6 MB for annotations.

In [ ]:
import json, urllib.request, zipfile, io, os, shutil
from tqdm import tqdm

IMG_DIR = WORK / "val2017"
IMG_DIR.mkdir(exist_ok=True)
ANN_DIR = WORK / "annotations"
ANN_DIR.mkdir(exist_ok=True)

with open(ROOT / "sampled_image_ids.json") as f:
    sampled_ids = json.load(f)
print(f"Sampled ids: {len(sampled_ids)} (first 5: {sampled_ids[:5]})")

missing = []
for iid in tqdm(sampled_ids, desc="val2017 images"):
    fn = f"{int(iid):012d}.jpg"
    out = IMG_DIR / fn
    if out.exists() and out.stat().st_size > 1024:
        continue
    url = f"http://images.cocodataset.org/val2017/{fn}"
    try:
        urllib.request.urlretrieve(url, out)
    except Exception as e:
        missing.append((iid, str(e)))
print("images on disk:", len(list(IMG_DIR.glob("*.jpg"))), "  missing:", len(missing))

ANN_PATH = ANN_DIR / "instances_val2017.json"
if not ANN_PATH.exists():
    ann_zip = WORK / "annotations_trainval2017.zip"
    if not ann_zip.exists():
        urllib.request.urlretrieve(
            "http://images.cocodataset.org/annotations/annotations_trainval2017.zip",
            ann_zip,
        )
    with zipfile.ZipFile(ann_zip) as z:
        z.extract("annotations/instances_val2017.json", WORK)
print("annotations:", ANN_PATH, ANN_PATH.exists())

Sampled ids: 500 (first 5: [776, 785, 872, 1353, 1532])


val2017 images: 100%|██████████| 500/500 [10:19<00:00,  1.24s/it]


images on disk: 500   missing: 0
annotations: /content/trackc_run/annotations/instances_val2017.json True


## 3. Load LLaVA-1.5-7B

Set `USE_4BIT = True` on a T4 (16 GB). Set `USE_4BIT = False` on H100/A100.

Eager attention is required either way because our VEGAS hook reshapes hidden states
over the 576 image-token span. In the 4-bit path the ViT is kept at fp16 so the
[CLS]→patch attention readout is numerically exact.

In [ ]:
!pip uninstall -y Pillow
!pip install -q --no-cache-dir "Pillow==11.3.0"
!python -c "import PIL, PIL._typing; print('Pillow', PIL.__version__, '| has _Ink:', hasattr(PIL._typing,'_Ink'))"

Found existing installation: pillow 12.2.0
Uninstalling pillow-12.2.0:
  Successfully uninstalled pillow-12.2.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 176.2 MB/s eta 0:00:00
Pillow 11.3.0 | has _Ink: False


In [ ]:
import torch
from trackc.vegas import VEGASRunner, VEGASConfig

assert torch.cuda.is_available(), "Use a GPU runtime (Colab → Runtime → Change runtime type)."
gpu_name = torch.cuda.get_device_name(0)
total_vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU: {gpu_name}  ({total_vram:.1f} GB)")

# Auto-select 4-bit on small GPUs.
USE_4BIT = total_vram < 24.0   # T4, L4 → 4-bit; A100/H100 → fp16
print("USE_4BIT =", USE_4BIT)

runner = VEGASRunner(
    model_id="llava-hf/llava-1.5-7b-hf",
    torch_dtype=torch.float16,
    load_in_4bit=USE_4BIT,
)
print("Loaded. Device:", runner.device,
      "| peak VRAM so far:",
      f"{torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")

GPU: Tesla T4  (14.6 GB)
USE_4BIT = True


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

...

model-00003-of-00003.safetensors:   0%|          | 0.00/4.18G [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

Loaded. Device: cuda:0 | peak VRAM so far: 6.46 GB


In [ ]:
# Hot-patch the runner for transformers>=4.47 layer paths.
import types

def _get_llm_layers(self):
    llm = self.model.language_model
    if hasattr(llm, "layers"):
        return llm.layers              # transformers >= 4.47
    return llm.model.layers            # transformers 4.37.x

def _install_hooks(self):
    handles = []
    for idx in self._cfg.steering_layers:
        layers = self._get_llm_layers()
        if 0 <= idx < len(layers):
            h = layers[idx].register_forward_pre_hook(
                self._make_attn_hook(idx), with_kwargs=True
            )
            handles.append(h)
    return handles

runner._get_llm_layers = types.MethodType(_get_llm_layers, runner)
runner._install_hooks  = types.MethodType(_install_hooks,  runner)

# Sanity check: we should find 32 decoder layers (LLaMA-7B).
print("llm.layers:", len(runner._get_llm_layers()))

llm.layers: 32


In [ ]:
# Hot-patch: correct the prefill-vs-decode guard in the VEGAS attention hook.
import types

def _make_attn_hook(self, layer_idx):
    cfg = self._cfg
    def hook(module, args, kwargs):
        if self._cls_attn is None or self._img_token_slice is None:
            return None
        hs = args[0] if len(args) > 0 else kwargs.get("hidden_states")
        if hs is None:
            return None
        sl = self._img_token_slice
        # Only rewrite during prefill, when the sequence actually contains
        # the 576 image tokens. During decode hs.size(1) == 1 -> skip.
        if hs.size(1) < sl.stop:
            return None
        lam = cfg.steering_weight
        if cfg.adaptive and self._vabe_state["gate"]:
            lam = min(1.0, lam * cfg.adaptive_scale)
        cls = self._cls_attn.to(hs.device, dtype=hs.dtype)
        w = cls / cls.mean().clamp_min(1e-8)
        w = (1.0 - lam) + lam * w
        new_hs = hs.clone()
        new_hs[:, sl, :] = hs[:, sl, :] * w.view(1, -1, 1)
        if len(args) > 0:
            return (new_hs,) + args[1:], kwargs
        kwargs["hidden_states"] = new_hs
        return args, kwargs
    return hook

runner._make_attn_hook = types.MethodType(_make_attn_hook, runner)
print("patched")

patched


In [ ]:
import types, torch

def _make_attn_hook(self, layer_idx):
    cfg = self._cfg
    def hook(module, args, kwargs):
        if self._cls_attn is None or self._img_token_slice is None:
            return None
        hs = args[0] if len(args) > 0 else kwargs.get("hidden_states")
        if hs is None:
            return None
        sl = self._img_token_slice
        if hs.size(1) < sl.stop:
            return None  # decode step, skip

        lam = cfg.steering_weight
        if cfg.adaptive and self._vabe_state["gate"]:
            lam = min(1.0, lam * cfg.adaptive_scale)

        cls = self._cls_attn.to(hs.device, dtype=hs.dtype)          # (576,), sums to 1
        w_rel = cls / cls.mean().clamp_min(1e-8)                    # mean = 1, peaky
        w_rel = torch.sqrt(w_rel.clamp_min(1e-6))                   # compress peaks
        w_rel = w_rel / w_rel.mean().clamp_min(1e-8)                # re-center to mean 1
        w_rel = w_rel.clamp(0.5, 2.0)                               # hard safety clip
        w = (1.0 - lam) + lam * w_rel                               # in [1-0.5*lam, 1+lam]

        new_hs = hs.clone()
        new_hs[:, sl, :] = hs[:, sl, :] * w.view(1, -1, 1)
        if len(args) > 0:
            return (new_hs,) + args[1:], kwargs
        kwargs["hidden_states"] = new_hs
        return args, kwargs
    return hook

runner._make_attn_hook = types.MethodType(_make_attn_hook, runner)
print("patched (bounded scale)")

patched (bounded scale)


## 4. Smoke test — 3 images

Before burning through all 500, caption 3 random images with **VEGAS off** (vanilla)
and **VEGAS on**, so you can eyeball whether the hook is taking effect. The vanilla
captions should match the style of Track A's `chair_labels.csv`.

In [ ]:
import random
from PIL import Image

PROMPT = "USER: <image>\nDescribe this image.\nASSISTANT:"

random.seed(0)
smoke_ids = random.sample(sampled_ids, 3)
print("Smoke-test image_ids:", smoke_ids)

cfg_off = VEGASConfig(steering_weight=0.0, adaptive=False, max_new_tokens=96)
cfg_on  = VEGASConfig(steering_weight=0.5, adaptive=True,  max_new_tokens=96)

for iid in smoke_ids:
    path = IMG_DIR / f"{int(iid):012d}.jpg"
    img = Image.open(path).convert("RGB")
    cap_off = runner.caption_one(img, PROMPT, cfg_off)
    cap_on  = runner.caption_one(img, PROMPT, cfg_on)
    print(f"\n--- image {iid} ---")
    print("vanilla :", cap_off)
    print("VEGAS   :", cap_on)

Smoke-test image_ids: [514376, 232244, 453001]

--- image 514376 ---
vanilla : The image features a blue and white train traveling down a street, passing by a tall building. The train is positioned in the middle of the scene, with the building on the left side.

There are several traffic lights visible in the image, with one located near the left edge, another in the middle, and the third on the right side. A car can be seen on the right side of the image, likely waiting for the train to pass.
VEGAS   : The image features a blue and white passenger train traveling down a street in a city. The train is positioned in the middle of the scene, with buildings on either side of the street.

There are several traffic lights visible in the image, with one located near the left edge, another in the middle, and the third on the right side. Additionally, there are two benches placed along the street, one near the left edge and the other closer to the center

--- image 232244 ---
vanilla : The ima

## 5. Full captioning sweep over all 500 images

We caption every image **twice** on the same model: once vanilla, once with VEGAS on.
Having both runs come from *the same weights* is important when `USE_4BIT=True`,
because Track A's `chair_labels.csv` was produced at fp16 — mixing fp16-vanilla with
4-bit-VEGAS would bias the Δ by the quantization noise.

Writes (in your Drive Track A folder):

- `captions_vanilla_4bit.csv` (or `_fp16`) — 500 × (image_id, caption)
- `captions_vegas.csv` — same shape

Checkpoints every **10 images** so a T4 idle disconnect costs you at most 10 samples.

In [ ]:
import pandas as pd
from tqdm import tqdm
from PIL import Image

TAG = "4bit" if USE_4BIT else "fp16"
VANILLA_CSV = ROOT / f"captions_vanilla_{TAG}.csv"
VEGAS_CSV   = ROOT / "captions_vegas.csv"

CFG_VANILLA = VEGASConfig(
    steering_weight=0.0, adaptive=False,
    max_new_tokens=96 if USE_4BIT else 128, do_sample=False,
)
CFG_VEGAS = VEGASConfig(
    steering_layers=(14, 15), steering_weight=0.5,
    adaptive=True, adaptive_scale=1.5, vabe_threshold=0.5,
    max_new_tokens=96 if USE_4BIT else 128, do_sample=False,
)

CHECKPOINT_EVERY = 10

def _run_pass(out_csv, cfg, label):
    done = {}
    if out_csv.exists():
        try:
            prev = pd.read_csv(out_csv)
            done = dict(zip(prev["image_id"].astype(int), prev["caption"].astype(str)))
            print(f"[{label}] resuming: {len(done)}/{len(sampled_ids)} already captioned.")
        except Exception as e:
            print(f"[{label}] could not resume:", e)

    records = [{"image_id": i, "caption": c} for i, c in done.items()]
    pbar = tqdm(sampled_ids, desc=f"{label} 500", ncols=90)
    for idx, iid in enumerate(pbar):
        if int(iid) in done:
            continue
        path = IMG_DIR / f"{int(iid):012d}.jpg"
        if not path.exists():
            print(f"[{label}] missing image {path}, skipping")
            continue
        try:
            img = Image.open(path).convert("RGB")
            cap = runner.caption_one(img, PROMPT, cfg)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            print(f"[{label}] OOM on {iid}, retrying once after cache clear")
            cap = runner.caption_one(img, PROMPT, cfg)
        records.append({"image_id": int(iid), "caption": cap})
        if (idx + 1) % CHECKPOINT_EVERY == 0:
            pd.DataFrame(records).drop_duplicates("image_id").to_csv(out_csv, index=False)

    df = (
        pd.DataFrame(records).drop_duplicates("image_id")
        .query("image_id in @sampled_ids")
        .sort_values("image_id").reset_index(drop=True)
    )
    df.to_csv(out_csv, index=False)
    print(f"[{label}] wrote {out_csv} ({len(df)} rows)")
    return df

print("\n>>> Pass 1/2: VANILLA")
_run_pass(VANILLA_CSV, CFG_VANILLA, "vanilla")

print("\n>>> Pass 2/2: VEGAS")
_run_pass(VEGAS_CSV, CFG_VEGAS, "vegas")


>>> Pass 1/2: VANILLA


vanilla 500: 100%|████████████████████████████████████| 500/500 [1:14:53<00:00,  8.99s/it]


[vanilla] wrote /content/drive/MyDrive/TrackA_Complete_Outputs/captions_vanilla_4bit.csv (500 rows)

>>> Pass 2/2: VEGAS


vegas 500: 100%|██████████████████████████████████████| 500/500 [1:15:36<00:00,  9.07s/it]

[vegas] wrote /content/drive/MyDrive/TrackA_Complete_Outputs/captions_vegas.csv (500 rows)


,image_id,caption
0,776,The image features a group of teddy bears sitt...
1,785,The image features a woman wearing a red jacke...
2,872,The image captures a baseball game in progress...
3,1353,The image features a group of children sitting...
4,1532,The image depicts a busy highway with several ...
...,...,...
495,579307,The image features a young boy standing on a b...
496,579902,The image features a person riding a motorcycl...
497,580294,The image features a woman standing in a kitch...
498,581615,The image features a public restroom with two ...


## 6. In-Colab ablation — stratified CHAIR, VEGAS vs vanilla

This is the main Track C deliverable. It re-merges `entropy_ranks.csv`,
`chair_labels.csv`, and the just-produced `captions_vegas.csv`; recomputes CHAIR on the
VEGAS captions against COCO ground truth; and prints the stratified table plus the
paired-bootstrap 95% CI on Δ mean CHAIR_i.

In [ ]:
from trackc.ablation import run

OUT_DIR = ROOT / "trackc_results"
result = run(
    entropy_csv=ROOT / "entropy_ranks.csv",
    baseline_csv=ROOT / "chair_labels.csv",
    out_dir=OUT_DIR,
    vegas_csv=VEGAS_CSV,
    coco_ann=ANN_PATH,
    bootstrap_iters=1000,
    # On T4 (4-bit), use the in-notebook 4-bit vanilla captions as baseline
    # so vanilla and VEGAS are at matched quantization.
    vanilla_captions_csv=(VANILLA_CSV if USE_4BIT else None),
)
print("\nWrote:", OUT_DIR)
for p in sorted(OUT_DIR.iterdir()):
    print(" -", p.name)

loading annotations into memory...
Done (t=0.62s)
creating index...
index created!
loading annotations into memory...
Done (t=0.55s)
creating index...
index created!

=== Track C — vanilla vs VEGAS (mean CHAIR_i by stratum) ===
       method         stratum   n  mean_chair_i  frac_hallucinating  sem_chair_i
vanilla_llava     low_entropy 125      0.173694               0.352     0.024630
vanilla_llava         mid_low 125      0.183396               0.464     0.019913
vanilla_llava        mid_high 125      0.227265               0.488     0.024431
vanilla_llava    high_entropy 125      0.195483               0.400     0.024737
vanilla_llava    binary:clean 250      0.178545               0.408     0.015808
vanilla_llava binary:degraded 250      0.211374               0.444     0.017378
vanilla_llava             all 500      0.194959               0.426     0.011757
        vegas     low_entropy 125      0.156264               0.336     0.023183
        vegas         mid_low 125      0.18

## 7. What to send back

From Drive, download (or just confirm the paths for me):

- `captions_vegas.csv` — VEGAS captions (500 rows)
- `captions_vanilla_4bit.csv` — **only present if you ran on T4**; matched-quantization vanilla baseline
- `trackc_results/merged_vegas.csv`
- `trackc_results/merged_baseline.csv`
- `trackc_results/ablation_summary.csv`
- `trackc_results/vegas_minus_vanilla_delta.csv`
- `trackc_results/bootstrap_delta.csv`

The local analysis step turns these into `TrackC_Report.docx`. On T4 the report will
note the 4-bit quantization in Methods.

# Track C — VEGAS × entropy ablation (CS 639)

**Inputs from Track A (same folder / Drive):**
- `entropy_ranks.csv` — per-image ViT rollout entropy + `entropy_group`
- `chair_labels.csv` — vanilla LLaVA captions + CHAIR scores

**What this notebook does**
1. **Stratified table (vanilla)** — mean CHAIR and hallucination rate per entropy stratum (always runnable).
2. **With VEGAS** — after you produce `captions_vegas.csv` (`image_id`, `caption`) using a VEGAS implementation, merge with entropy and compare CHAIR vs vanilla.

**Paper (implementation reference):** [VEGAS arXiv:2512.12089](https://arxiv.org/abs/2512.12089). For LLaVA-1.5 the authors use the **vision encoder final-layer [CLS] attention** over image tokens, injected at **LLM layers 14 and 15** (0-indexed), plus adaptive logits steering (see §5.1 *Implementation Details*).


## 1 — Setup (Colab: GPU runtime optional for §1–2 only)

Local: `python -m venv .venv && source .venv/bin/activate && pip install -r requirements-trackc.txt`


In [ ]:
# Colab-only installs (comment out if local venv already has deps)
# !pip install -q pandas pycocotools


## 2 — Paths + run stratified analysis (vanilla baseline)

Set `ROOT` to the folder that contains `entropy_ranks.csv` and `chair_labels.csv`.


In [ ]:
import sys
from pathlib import Path

# Folder that contains `entropy_ranks.csv`, `chair_labels.csv`, and the `trackc/` package
ROOT = Path("/content/drive/MyDrive/CS639_TrackA")  # Colab; use Path(".") for local clone
sys.path.insert(0, str(ROOT))

from trackc.ablation import run

ENTROPY_CSV = ROOT / "entropy_ranks.csv"
BASELINE_CSV = ROOT / "chair_labels.csv"
OUT_DIR = ROOT / "trackc_results"

run(ENTROPY_CSV, BASELINE_CSV, OUT_DIR, vegas_csv=None, coco_ann=None)
print("Wrote:", OUT_DIR / "ablation_summary.csv", OUT_DIR / "merged_baseline.csv")


## 3 — Full comparison after VEGAS captions exist

1. Run **VEGAS** decoding on the **same 500** `image_id`s (same prompt as Track A: `USER: <image>\nDescribe this image.\nASSISTANT:` unless your team standardizes otherwise).
2. Save **`captions_vegas.csv`** with columns `image_id,caption`.
3. Download **`instances_val2017.json`** (COCO) so CHAIR can be recomputed on new captions.

Then run the cell below.


In [ ]:
from trackc.ablation import run

COCO_ANN = ROOT / "coco" / "annotations" / "instances_val2017.json"  # adjust
VEGAS_CSV = ROOT / "captions_vegas.csv"

if VEGAS_CSV.exists() and COCO_ANN.exists():
    run(ENTROPY_CSV, BASELINE_CSV, OUT_DIR, vegas_csv=VEGAS_CSV, coco_ann=COCO_ANN)
    print("See:", OUT_DIR / "vegas_minus_vanilla_delta.csv")
else:
    print("Skip: add captions_vegas.csv and COCO annotations path, then re-run.")


## 4 — Implementing VEGAS (not included as turnkey code)

The public **VEGAS** method is *inference-time*: replace / steer **LLM** visual-token attention in **middle layers** using **vision-encoder** attention, then **adaptively** blend logits (paper: higher steering weight when **VABE** is high).

**Practical path for the course project**
- Watch the paper repo / authors’ release; or
- Reimplement from §3–4 + Appendix in the PDF, matching **layers 14–15** and **LLaVA-1.5-7B** as in §5.1.

**Track C success criteria (from sprint doc)**  
Report a table: mean CHAIR (and/or hallucination rate) for **vanilla vs VEGAS**, split by **low vs high entropy** (from Track A). Test whether VEGAS **hurts** when ViT attention was already **high-entropy** (degraded).
